# XP-AcceptanceAI — Final Evaluation
MSCS811 | Independent synthetic cases F01–F12 (C01–C03 excluded).

Run in order. The first three code cells write the frozen implementation and the next cell validates the benchmark without API calls. Use the same Gemini free-tier key and model. Run one case per batch initially. Results are checkpointed in a dedicated local Colab folder; export a ZIP after each batch to retain them across runtime resets. Historical results are provided separately. Never overwrite V1/V2 pilot results.

Methods: A full regeneration, B direct update, C direct + execution validation, D selective + execution validation (V1), C2 C + additive coverage review (control), D2 D + identical additive coverage review (V2). C2/D2 reuse their respective exact base suite, not a new independent generation. Target: 12 cases × 3 repeats × 6 methods = 216 evaluated rows; 144 base runs and 72 dependent review records. Stop on provider errors and resume missing trials; do not change model or protocol mid-study. Synthetic faults and reference labels are hidden from model prompts.

In [ ]:
%%writefile pilot.py
"""XP-AcceptanceAI: reproducible API acceptance-test maintenance pilot.
Synthetic requirement changes on a standalone Conduit-inspired HTTP service.
No model calls are made by smoke(). Python 3.10+; standard library only for smoke.
"""
import argparse, copy, csv, json, os, time, threading, urllib.request, urllib.error
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from contextlib import contextmanager

CASES = {
 'C01': {'old':'Article list items include the full body.', 'new':'Article list items omit body; single-article responses still include body.', 'unchanged':'Article detail returns body; authenticated article authors can delete; unauthenticated comments are rejected; valid comments can be added.', 'faults':['list_body_leak','detail_body_missing','comment_auth_bypass']},
 'C02': {'old':'Any authenticated user can delete an article.', 'new':'Only the article author can delete it; other authenticated users receive 403.', 'unchanged':'Unauthenticated deletion is rejected; author deletion works; article list and detail include body; comments require authentication.', 'faults':['delete_any_user','delete_author_denied','comment_auth_bypass']},
 'C03': {'old':'Authenticated users can post comments with any nonempty body.', 'new':'Authenticated users can post comments with at least 3 characters; shorter comments receive 422.', 'unchanged':'A 3-character comment is accepted; comments require authentication; article detail and list include body; author deletion works.', 'faults':['comment_short_allowed','comment_boundary_wrong','detail_body_missing']},
}
API = '''Baseline business rules (the new requirement overrides the relevant rule): both article list and detail include body; any authenticated user may delete an article; anonymous deletion returns 401; successful deletion removes the article; only authenticated users may comment; empty comments are rejected, any nonempty string accepted; comment response echoes text and author. Fresh state per test: article slug hello, author alice, body Hello world. Tokens alice and bob; use auth field as token or omit for anonymous. GET /api/articles -> 200 {articles:[{slug,author,body?}]}; GET /api/articles/hello -> 200 {article:{slug,author,body}} or 404. DELETE /api/articles/hello -> 204, or 401 anonymous, 403 forbidden, 404 missing. POST /api/articles/hello/comments with JSON {comment:{body: string}} -> 201 {comment:{body,author}}, 401 anonymous, 422 invalid. Test DSL: {id, steps:[{method,path,auth?,json?,expect:{status, checks:[{path:[string or integer,...],op:"eq"|"absent"|"exists",value?}]}}]}. Checks navigate response JSON; status always required. No arbitrary code or external URLs. Each test has fresh seeded state; multiple steps in a test share state. IDs stable for existing tests.'''

def step(method,path,status,auth=None,body=None,checks=()):
 d={'method':method,'path':path,'expect':{'status':status,'checks':list(checks)}}
 if auth: d['auth']=auth
 if body is not None: d['json']={'comment':{'body':body}}
 return d

def chk(path,op,value=None):
 d={'path':path,'op':op}
 if op=='eq':d['value']=value
 return d

def test(id,*steps):return {'id':id,'steps':list(steps)}
P='/api/articles/hello'; L='/api/articles'; C=P+'/comments'
OLD=[test('T01',step('GET',L,200,checks=[chk(['articles',0,'body'],'eq','Hello world')])),
 test('T02',step('GET',P,200,checks=[chk(['article','body'],'eq','Hello world')])),
 test('T03',step('DELETE',P,204,'alice'),step('GET',P,404)),
 test('T04',step('DELETE',P,204,'bob'),step('GET',P,404)),
 test('T05',step('DELETE',P,401)),
 test('T06',step('POST',C,401,body='hello')),
 test('T07',step('POST',C,201,'bob','x',[chk(['comment','body'],'eq','x')])),
 test('T08',step('POST',C,201,'alice','hello',[chk(['comment','body'],'eq','hello')])),
 test('T09',step('POST',C,422,'alice',''))]
# Reference oracle is separately specified, never passed to model.
IMPACT={'C01':{'T01'},'C02':{'T04'},'C03':{'T07'}}

def reference(case):
 s=copy.deepcopy(OLD)
 if case=='C01':s[0]=test('T01',step('GET',L,200,checks=[chk(['articles',0,'body'],'absent'),chk(['articles',0,'slug'],'eq','hello')]))
 if case=='C02':s[3]=test('T04',step('DELETE',P,403,'bob'),step('GET',P,200,checks=[chk(['article','slug'],'eq','hello')]))
 if case=='C03':
  s[6]=test('T07',step('POST',C,422,'bob','x'))
  s.append(test('T10',step('POST',C,201,'bob','abc',[chk(['comment','body'],'eq','abc')])))
  s.append(test('T11',step('POST',C,422,'bob','ab')))
 return s

class Handler(BaseHTTPRequestHandler):
 def log_message(self,*a):pass
 def do_GET(self):self.handle_api()
 def do_DELETE(self):self.handle_api()
 def do_POST(self):self.handle_api()
 def reply(self,status,payload=None):
  b=json.dumps(payload or {}).encode() if status!=204 else b''
  self.send_response(status);self.send_header('Content-Type','application/json');self.send_header('Content-Length',str(len(b)));self.end_headers();self.wfile.write(b)
 def handle_api(self):
  s=self.server; case=s.case; new=s.new; fault=s.fault; auth=self.headers.get('Authorization','').removeprefix('Token ')
  auth=auth if auth in ('alice','bob') else None
  article={'slug':'hello','author':'alice','body':'Hello world'}
  if self.command=='GET' and self.path==L:
   if new and case=='C01' and fault!='list_body_leak':article.pop('body')
   return self.reply(200,{'articles':[article] if s.exists else []})
  if self.path not in (P,C) or not s.exists:return self.reply(404)
  if self.command=='GET' and self.path==P:
   if fault=='detail_body_missing':article.pop('body')
   return self.reply(200,{'article':article})
  if self.command=='DELETE' and self.path==P:
   if not auth:return self.reply(401)
   if new and case=='C02' and auth!='alice' and fault!='delete_any_user':return self.reply(403)
   if fault=='delete_author_denied' and auth=='alice':return self.reply(403)
   s.exists=False;return self.reply(204)
  if self.command=='POST' and self.path==C:
   if not auth and fault!='comment_auth_bypass':return self.reply(401)
   try:body=json.loads(self.rfile.read(int(self.headers.get('Content-Length','0'))))['comment']['body']
   except Exception:return self.reply(422)
   minimum=3 if new and case=='C03' else 1
   if fault=='comment_short_allowed':minimum=1
   if fault=='comment_boundary_wrong':minimum=4
   if not isinstance(body,str) or len(body)<minimum:return self.reply(422)
   return self.reply(201,{'comment':{'body':body,'author':auth or 'anonymous'}})
  return self.reply(404)

@contextmanager
def server(case,new=True,fault=None):
 s=ThreadingHTTPServer(('127.0.0.1',0),Handler);s.case=case;s.new=new;s.fault=fault;s.exists=True
 t=threading.Thread(target=s.serve_forever,kwargs={'poll_interval':0.01},daemon=True);t.start()
 try:yield s
 finally:s.shutdown();s.server_close();t.join()

def validate(suite):
 if not isinstance(suite,list) or not 1<=len(suite)<=30:raise ValueError('suite must contain 1-30 tests')
 ids=set()
 for t in suite:
  if not isinstance(t,dict) or not isinstance(t.get('id'),str) or not t['id'] or t['id'] in ids:raise ValueError('unique test IDs required')
  ids.add(t['id'])
  if not isinstance(t.get('steps'),list) or not 1<=len(t['steps'])<=8:raise ValueError('1-8 steps required')
  for x in t['steps']:
   if x.get('method') not in ('GET','POST','DELETE') or x.get('path') not in (P,L,C):raise ValueError('invalid method/path')
   if x.get('auth') not in (None,'alice','bob'):raise ValueError('invalid auth')
   e=x.get('expect',{});status=e.get('status')
   if type(status)!=int or not 100<=status<=599:raise ValueError('expected status required')
   if not isinstance(e.get('checks',[]),list):raise ValueError('checks must be list')
   for c in e.get('checks',[]):
    if not isinstance(c.get('path'),list) or not c['path'] or any(type(k) not in (str,int) for k in c['path']) or c.get('op') not in ('eq','exists','absent') or (c.get('op')=='eq' and 'value' not in c):raise ValueError('invalid check')
 return suite

def run_suite(suite,case,new=True,fault=None):
 validate(suite); failures=[]; passed=0
 with server(case,new,fault) as s:
  for t in suite:
   s.exists=True
   for i,x in enumerate(t['steps']):
    req=urllib.request.Request(f'http://127.0.0.1:{s.server_port}'+x['path'],method=x['method'],data=json.dumps(x['json']).encode() if 'json'in x else None,headers={'Content-Type':'application/json',**({'Authorization':'Token '+x['auth']} if x.get('auth') else {})})
    try:
     try:r=urllib.request.urlopen(req,timeout=3)
     except urllib.error.HTTPError as e:r=e
     with r:status=r.code;b=r.read();data=json.loads(b) if b else {}
     exp=x['expect']; problems=[]
     if status!=exp['status']:problems.append(f'status expected {exp["status"]}, got {status}')
     for c in exp.get('checks',[]):
      v=data;exists=True
      try:
       for k in c['path']:v=v[k]
      except (KeyError,IndexError,TypeError):exists=False
      if not ((c['op']=='absent' and not exists) or (c['op']=='exists' and exists) or (c['op']=='eq' and exists and v==c['value'])):problems.append(f'check {c} failed (actual={v if exists else "MISSING"})')
     if problems:failures.append({'id':t['id'],'step':i,'errors':problems});break
    except Exception as e:raise RuntimeError(f'Infrastructure/runner error in {t["id"]}: {e}') from e
   else:passed+=1
 return {'passed':passed,'total':len(suite),'failures':failures,'pass':not failures}

def evaluate(suite,case):
 correct=run_suite(suite,case); faults=CASES[case]['faults']; killed=[]
 # Invalid suites receive zero credit; never count a preexisting failure as a fault detection.
 if correct['pass']:
  for f in faults:
   if not run_suite(suite,case,fault=f)['pass']:killed.append(f)
 new_faults=set(faults[:2] if case=='C03' else faults[:1]); regression_faults=set(faults)-new_faults
 old={t['id']:t for t in OLD}; actual={t['id']:t for t in suite}; changed={k for k,v in old.items() if actual.get(k)!=v}; expected=IMPACT[case]
 return {'correct_pass':correct['pass'],'passed_tests':correct['passed'],'total_tests':len(suite),'killed':killed,'new_faults_killed':len(new_faults.intersection(killed)),'new_faults_total':len(new_faults),'regression_faults_killed':len(regression_faults.intersection(killed)),'regression_faults_total':len(regression_faults),'changed_existing':len(changed),'unnecessary_changes':len(changed-expected),'added_tests':len(set(actual)-set(old)),'failures':correct['failures']}

def smoke(out):
 rows=[]
 for case in CASES:
  before=run_suite(OLD,case,new=False);assert before['pass'],before
  stale=run_suite(OLD,case);assert not stale['pass'],stale
  ref=reference(case);e=evaluate(ref,case);assert e['correct_pass'] and len(e['killed'])==3,e
  rows.append({'case':case,'old_on_old':before['passed'],'old_on_new':stale['passed'],'reference_on_new':e['passed_tests'],'reference_total':e['total_tests'],'reference_faults_killed':len(e['killed']),'faults':e['killed']})
 Path(out).mkdir(parents=True,exist_ok=True);Path(out,'smoke.json').write_text(json.dumps(rows,indent=2));print(json.dumps(rows,indent=2));return rows

# Actual LLM experiment. Model receives public requirements and API contract only,
# not reference tests, impact labels, fault names or mutant execution results.
def model_call(client,model,prompt,logs):
 start=time.perf_counter();r=client.chat.completions.create(model=model,messages=[{'role':'system','content':'You maintain executable API acceptance tests. Return a JSON object only, no Markdown.'},{'role':'user','content':prompt}],response_format={'type':'json_object'})
 raw=r.choices[0].message.content;u=r.usage
 logs.append({'prompt':prompt,'raw':raw,'seconds':time.perf_counter()-start,'tokens':getattr(u,'total_tokens',None),'model':r.model});return json.loads(raw)

def experiment(out,model,repeats=1,repairs=2,client=None):
 # Only client configuration changed for publication; experiment logic is unchanged.
 if client is None:
  from openai import OpenAI
  key=os.environ.get('GEMINI_API_KEY')
  if not key:raise RuntimeError('Set GEMINI_API_KEY before running the LLM experiment.')
  client=OpenAI(api_key=key,base_url='https://generativelanguage.googleapis.com/v1beta/openai/',max_retries=0,timeout=120)
 out=Path(out);out.mkdir(parents=True,exist_ok=True);rows=[]
 for case,c in CASES.items():
  public={k:v for k,v in c.items() if k!='faults'}; common=API+'\nRequirements: '+json.dumps(public)+'\nReturn JSON {"tests": [...]} using DSL. Preserve stable IDs for existing tests; use new IDs for additions. Do not weaken checks.'
  for rep in range(repeats):
   for method in ('A','B','C','D'):
    logs=[];start=time.perf_counter();selected=None;row={'case':case,'repeat':rep+1,'method':method,'model':model};suite=None
    try:
     if method=='D':
      p=common+'\nExisting tests: '+json.dumps(OLD)+'\nFirst return JSON {"update_ids":[...],"reasons":{...}} identifying existing tests that need edits, not merely related tests. New tests can be added later.'
      plan=model_call(client,model,p,logs);selected=set(plan['update_ids']);assert selected<=set(t['id'] for t in OLD),'unknown IDs'
      allowed=[t for t in OLD if t['id'] in selected];locked=[t for t in OLD if t['id'] not in selected];base=common+'\nEditable tests: '+json.dumps(allowed)+'\nProtected tests (read-only): '+json.dumps(locked)+'\nReturn only updated editable tests (all selected IDs) and any added tests, never protected IDs.'
      def merge(data):
       changed=data['tests'];
       if changed:validate(changed)
       elif changed!=[]:raise ValueError('tests must be list')
       ids={t['id'] for t in changed};assert selected<=ids and not ids.intersection(t['id'] for t in locked),'protected or missing IDs';return validate(locked+changed)
      # An empty patch is legal; evaluation detects missed changes.
     else:
      base=common+ ('\nRegenerate entire suite from requirements, without old tests.' if method=='A' else '\nExisting tests: '+json.dumps(OLD)+'\nUpdate complete suite directly; preserve unaffected behavior.')
      def merge(data):return validate(data['tests'])
     for attempt in range((repairs if method in ('C','D') else 0)+1):
      try:
       data=model_call(client,model,base,logs);suite=merge(data)
       if method not in ('C','D'):break
       result=run_suite(suite,case)
       if result['pass']:break
       base+='\nPrevious output: '+json.dumps(data)+'\nCorrect-version failures: '+json.dumps(result['failures'])+'\nRepair only legitimate issues; follow original constraints.'
      except (ValueError,KeyError,TypeError,AssertionError) as e:
       suite=None;base+='\nInvalid previous output: '+str(e)+'\nReturn valid JSON in required schema.'
     if suite is None:raise ValueError('No valid test suite')
     row.update(evaluate(suite,case));row['status']='evaluated'
     if method=='A':
      for k in ('changed_existing','unnecessary_changes','added_tests'):row[k]=None  # IDs not comparable for fresh regeneration
     if selected is not None:
      tp=len(selected&IMPACT[case]);row.update(impact_precision=tp/len(selected) if selected else 0,impact_recall=tp/len(IMPACT[case]))
    except Exception as e:row.update(status='error',error=str(e),correct_pass=False,new_faults_killed=0,regression_faults_killed=0)
    row.update(seconds=time.perf_counter()-start,tokens=sum(l['tokens'] or 0 for l in logs),calls=len(logs));rows.append(row)
    (out/f'{case}_{method}_{rep+1}.json').write_text(json.dumps({'metrics':row,'suite':suite,'calls':logs},indent=2));print(case,method,rep+1,row['status'])
    (out/'results.json').write_text(json.dumps(rows,indent=2))
 fields=sorted(set().union(*(r.keys() for r in rows)))
 with (out/'results.csv').open('w',newline='') as f:w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
 return rows

if __name__=='__main__':
 p=argparse.ArgumentParser();p.add_argument('mode',choices=['smoke','llm']);p.add_argument('--out',default='results');p.add_argument('--model',default=os.getenv('XP_MODEL','gemini-3.5-flash-lite'));p.add_argument('--repeats',type=int,default=1);p.add_argument('--repairs',type=int,default=2);a=p.parse_args();smoke(a.out) if a.mode=='smoke' else experiment(a.out,a.model,a.repeats,a.repairs)


In [ ]:
%%writefile coverage_v2.py
"""Pilot V2: paired, additive requirement-coverage review of archived V1 D suites.
Model self-reports coverage; independent fault evaluation is never model feedback.
"""
import copy, csv, json, time
from pathlib import Path
from pilot import API, CASES, validate, run_suite, evaluate, model_call

RULES = '''Review the executable suite against the public requirement change and retained behavior.
Derive test obligations from the requirements: include positive/negative outcomes, exact boundaries and adjacent values when relevant, authorization roles, and relevant state/response checks. Do not assume that passing tests are complete. Inspect actual inputs and expected assertions, not only test names.
Keep every existing test unchanged. Add only necessary tests with fresh IDs beginning V2_. Use the DSL and executable endpoints. Never copy an existing ID. Do not use external code. If no additions are necessary, return an empty tests array.
Return one JSON object with keys obligations and tests. Each obligation has id, description, status (covered or missing), test_ids (array of IDs in the final suite), and evidence (short explanation citing actual inputs/assertions). For a missing obligation use an empty test_ids list. Include both already covered and newly covered obligations. tests contains additions only. Avoid duplicates and irrelevant checks.'''

def prompt_for(case, suite):
    public = {k:v for k,v in CASES[case].items() if k != 'faults'}
    return API + '\nPublic requirements: ' + json.dumps(public) + '\nExisting immutable tests: ' + json.dumps(suite) + '\n' + RULES

def merge_review(original, data):
    if not isinstance(data, dict): raise ValueError('Expected JSON object')
    additions=data.get('tests')
    if not isinstance(additions,list): raise ValueError('tests must be list')
    if additions: validate(additions)
    if any(not t['id'].startswith('V2_') for t in additions): raise ValueError('New IDs must begin V2_')
    merged=validate(copy.deepcopy(original)+copy.deepcopy(additions))
    obs=data.get('obligations')
    if not isinstance(obs,list) or not obs: raise ValueError('Nonempty obligations required')
    ids={t['id'] for t in merged}; oids=set(); missing=[]
    for o in obs:
        if not isinstance(o,dict) or not all(isinstance(o.get(k),str) and o[k].strip() for k in ('id','description','evidence')): raise ValueError('Invalid obligation')
        if o['id'] in oids: raise ValueError('Duplicate obligation ID')
        oids.add(o['id']); links=o.get('test_ids')
        if not isinstance(links,list) or any(not isinstance(k,str) or k not in ids for k in links): raise ValueError('Invalid obligation test IDs')
        if o.get('status')=='covered' and links: pass
        elif o.get('status')=='missing' and not links: missing.append(o['description'])
        else: raise ValueError('Invalid obligation coverage status')
    return merged, missing

def review_case(case, original, ask, max_repairs=2):
    original=copy.deepcopy(validate(original))
    if not run_suite(original,case)['pass']: raise ValueError('Archived V1 suite must pass correct version')
    prompt=prompt_for(case,original); feedback=''; attempts=[]; best=copy.deepcopy(original); accepted=[]; state='incomplete'
    for i in range(max_repairs+1):
        try:
            data=ask(prompt+feedback)
            candidate, missing=merge_review(original,data)
            result=run_suite(candidate,case)
            if not result['pass']:
                feedback='\nCorrect-version failures: '+json.dumps(result['failures'])+'\nPrevious response: '+json.dumps(data)+'\nRevise additions only; original remains immutable.'
                attempts.append({'attempt':i+1,'status':'execution_failed','failures':result['failures']}); continue
            best=candidate; accepted=data['obligations']
            attempts.append({'attempt':i+1,'status':'valid','reported_missing':missing})
            if not missing: state='model_reported_complete'; break
            feedback='\nPrevious response: '+json.dumps(data)+'\nReported missing obligations: '+json.dumps(missing)+'\nReturn a complete revised review and additions only.'
        except (ValueError,TypeError,KeyError,AttributeError,AssertionError) as exc:
            attempts.append({'attempt':i+1,'status':'invalid','error':str(exc)})
            feedback='\nInvalid response: '+str(exc)+'\nReturn valid JSON following the schema.'
        except Exception as exc:
            # Provider errors do not erase the already validated suite or earlier results.
            attempts.append({'attempt':i+1,'status':'provider_or_runner_error','error_type':type(exc).__name__}); state='provider_or_runner_error'; break
    assert best[:len(original)]==original
    return best, accepted, state, attempts

def paired_experiment(archive, client, model, out='llm_results_v2', max_repairs=2):
    out=Path(out);out.mkdir(parents=True,exist_ok=True); rows=[]
    for case, saved in archive.items():
        if saved['metrics']['model']!=model: raise ValueError('Model differs from archived V1; run a fresh paired baseline for fair comparison')
        original=saved['suite']; before=evaluate(original,case); logs=[]; start=time.perf_counter()
        suite, obligations, status, attempts=review_case(case,original,lambda p:model_call(client,model,p,logs),max_repairs)
        after=evaluate(suite,case); extra_seconds=time.perf_counter()-start
        extra_tokens=sum(x['tokens'] or 0 for x in logs); old=saved['metrics']
        for version, metrics in [('V1',before),('V2',after)]:
            is_v2=version=='V2'
            row={'case':case,'version':version,'model':model,'correct_pass':metrics['correct_pass'],'tests':metrics['total_tests'],'new_faults_killed':metrics['new_faults_killed'],'new_faults_total':metrics['new_faults_total'],'regression_faults_killed':metrics['regression_faults_killed'],'regression_faults_total':metrics['regression_faults_total'],'unnecessary_changes':metrics['unnecessary_changes'],'added_tests':metrics['added_tests'],'review_status':status if is_v2 else 'archived','seconds':old['seconds']+(extra_seconds if is_v2 else 0),'tokens':old['tokens']+(extra_tokens if is_v2 else 0),'calls':old['calls']+(len(logs) if is_v2 else 0),'review_seconds':extra_seconds if is_v2 else 0,'review_tokens':extra_tokens if is_v2 else 0,'review_calls':len(logs) if is_v2 else 0}
            rows.append(row)
        (out/f'{case}_V2.json').write_text(json.dumps({'case':case,'model':model,'v1_metrics':old,'v1_suite':original,'v2_suite':suite,'v2_metrics':after,'review_status':status,'obligations':obligations,'attempts':attempts,'calls':logs},indent=2))
        (out/'results.json').write_text(json.dumps(rows,indent=2))
        with (out/'results.csv').open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
        print(case,'V2',status,'| new faults',before['new_faults_killed'],'->',after['new_faults_killed'],'| tests',len(original),'->',len(suite),flush=True)
    return rows


In [ ]:
%%writefile final_study.py
"""Held-out synthetic requirement changes. No model calls in smoke().
The development pilot's C01-C03 are excluded. Six methods, paired C2/D2.
"""
import copy, csv, json, time, random, os, tempfile
from pathlib import Path
from contextlib import contextmanager
from http.server import ThreadingHTTPServer
import threading
import pilot as p
import coverage_v2 as review
P,L,C=p.P,p.L,p.C
step,chk,test=p.step,p.chk,p.test

def get(id,path,keys,value=None,op='eq',status=200):
 return test(id,step('GET',path,status,checks=[chk(keys,op,value)] if keys else []))
def post(id,body,status=201,auth='alice',checks=None):
 return test(id,step('POST',C,status,auth,body,checks if checks is not None else ([chk(['comment','body'],'eq',body)] if status==201 else [])))
def delete(id,auth,status,after=404,checks=()):
 return test(id,step('DELETE',P,status,auth,checks=checks),step('GET',P,after))
OLD=[get('T01',L,['articles',0,'body'],'Hello world'),get('T02',P,['article','body'],'Hello world'),delete('T03','alice',204),delete('T04','bob',204),test('T05',step('DELETE',P,401)),post('T06','hello',401,None),post('T07','x',auth='bob'),post('T08','hello'),post('T09','',422),get('T10',L,['articles',0,'author'],'alice'),get('T11',P,['article','author'],'alice'),post('T12','123456789'),post('T13','   '),post('T14','  hi  '),post('T15','hello',checks=[chk(['comment','author'],'eq','alice')]),post('T16','hello',auth='bob',checks=[chk(['comment','author'],'eq','bob')]),get('T17',P,['article','slug'],'hello'),get('T18',L,['articles',0,'slug'],'hello')]
# Public requirements only; fault and impact annotations are not model inputs.
SPECS=[
 ('F01','Article lists include author.','Article lists omit author; article detail still includes author.','T10'),
 ('F02','Article detail has no title field.','Article detail includes title equal to "Hello"; existing fields remain.',''),
 ('F03','Fetching a deleted article returns 404.','Fetching a deleted article returns 410; successful deletion still returns 204.','T03 T04'),
 ('F04','Anonymous deletion returns 401.','Anonymous deletion returns 403 and must leave the article intact.','T05'),
 ('F05','Anonymous comments return 401.','Anonymous comments with nonempty bodies are accepted (201), with author "guest"; empty bodies are rejected (422).','T06'),
 ('F06','Both authenticated users may comment.','Only alice may comment; bob receives 403 even for a valid body. Anonymous comments remain 401.','T07 T16'),
 ('F07','Any nonempty comment length is accepted.','Comment bodies must have 1 to 8 characters inclusive; longer bodies return 422.','T12'),
 ('F08','Whitespace-only nonempty comments are accepted.','Whitespace-only comments are rejected (422); other nonempty comments retain their original text.','T13'),
 ('F09','Comment response echoes the input body exactly.','Trim leading and trailing whitespace before storing and returning a comment; reject the result if empty (422).','T13 T14'),
 ('F10','Comment response contains body and author.','Successful comment response additionally includes length equal to the number of characters in the body.',''),
 ('F11','Empty comments return 422.','Empty string comments are accepted (201); missing or non-string body still returns 422.','T09'),
 ('F12','Successful deletion returns 204 with no body.','Successful deletion returns 200 and JSON {"deleted":true}; article is removed.','T03 T04'),
]
CASES={c:{'old':old,'new':new,'unchanged':'All baseline rules and response fields remain except the explicitly changed behavior.','faults':['rule_omitted','rule_incorrect','regression_detail_body']} for c,old,new,ids in SPECS}
IMPACT={c:set(ids.split()) for c,old,new,ids in SPECS}
API=p.API+''' Additional baseline details: list/detail include slug, author and body; no title field in detail; comment response has body and author, no length. Missing/non-string comment body -> 422 (after authorization); whitespace is not trimmed and whitespace-only strings are accepted. Deletion removes article; GET detail then returns 404. Baseline DELETE for both alice and bob is 204; anonymous 401. The new requirement overrides only affected rules. JSON booleans are true/false. To test missing/non-string comment body, set json explicitly in the DSL. All listed baseline rules are active unless overridden.''' 

class Handler(p.Handler):
 def handle_api(self):
  s=self.server; c=s.case if s.new and s.fault!='rule_omitted' else ''; wrong=s.fault=='rule_incorrect'; a=self.headers.get('Authorization','').removeprefix('Token '); a=a if a in ('alice','bob') else None
  article={'slug':'hello','author':'alice','body':'Hello world'}
  if self.command=='GET' and self.path==L:
   if c=='F01':
    if wrong:article['author']='guest'
    else:article.pop('author')
   return self.reply(200,{'articles':[article] if s.exists else []})
  if self.path not in (P,C) or not s.exists:
   return self.reply((409 if wrong else 410) if c=='F03' and self.command=='GET' and self.path==P else 404)
  if self.command=='GET' and self.path==P:
   if c=='F02':article['title']='Wrong' if wrong else 'Hello'
   if s.fault=='regression_detail_body':article.pop('body')
   return self.reply(200,{'article':article})
  if self.command=='DELETE' and self.path==P:
   if not a:
    if c=='F04' and wrong:s.exists=False
    return self.reply(403 if c=='F04' else 401)
   s.exists=False
   if c=='F12':return self.reply(200,{'deleted':not wrong})
   return self.reply(204)
  if self.command=='POST' and self.path==C:
   if not a and c!='F05':return self.reply(401)
   if a=='bob' and c=='F06':return self.reply(401 if wrong else 403)
   try:body=json.loads(self.rfile.read(int(self.headers.get('Content-Length','0'))))['comment']['body']
   except Exception:return self.reply(422)
   if not isinstance(body,str):return self.reply(422)
   if c=='F09':body=body.lstrip() if wrong else body.strip()
   if not body and c!='F11':return self.reply(422)
   if c=='F11' and wrong and not body:return self.reply(201,{'comment':{'body':'empty','author':a}})
   if c=='F07' and len(body)>(7 if wrong else 8):return self.reply(422)
   if c=='F08' and not body.strip():
    if not wrong or len(body)==1:return self.reply(422)
   result={'body':body,'author':a or ('anonymous' if wrong else 'guest')}
   if c=='F10':result['length']=len(body)+(1 if wrong else 0)
   return self.reply(201,{'comment':result})
  return self.reply(404)

@contextmanager
def server(case,new=True,fault=None):
 s=ThreadingHTTPServer(('127.0.0.1',0),Handler);s.case=case;s.new=new;s.fault=fault;s.exists=True
 t=threading.Thread(target=s.serve_forever,kwargs={'poll_interval':.01},daemon=True);t.start()
 try:yield s
 finally:s.shutdown();s.server_close();t.join()

def reference(case):
 s=copy.deepcopy(OLD); changes={}; adds=[]
 if case=='F01':changes['T10']=get('T10',L,['articles',0,'author'],op='absent')
 if case=='F02':adds=[get('R01',P,['article','title'],'Hello')]
 if case=='F03':changes={x:delete(x,a,204,410) for x,a in [('T03','alice'),('T04','bob')]}
 if case=='F04':changes['T05']=delete('T05',None,403,200)
 if case=='F05':changes['T06']=post('T06','hello',201,None,[chk(['comment','author'],'eq','guest'),chk(['comment','body'],'eq','hello')]);adds=[post('R01','',422,None)]
 if case=='F06':changes={'T07':post('T07','x',403,'bob'),'T16':post('T16','hello',403,'bob')}
 if case=='F07':changes['T12']=post('T12','123456789',422);adds=[post('R01','12345678')]
 if case=='F08':changes['T13']=post('T13','   ',422);adds=[post('R01',' ',422)]
 if case=='F09':changes={'T13':post('T13','   ',422),'T14':post('T14','  hi  ',201,checks=[chk(['comment','body'],'eq','hi')])}
 if case=='F10':adds=[post('R01','hello',checks=[chk(['comment','length'],'eq',5)])]
 if case=='F11':changes['T09']=post('T09','')
 if case=='F12':changes={x:delete(x,a,200,checks=[chk(['deleted'],'eq',True)]) for x,a in [('T03','alice'),('T04','bob')]}
 return [changes.get(x['id'],x) for x in s]+adds

# Reuse unchanged DSL and execution logic. Bind the expanded independent benchmark.
p.server=server;p.CASES=CASES;p.API=API;p.OLD=OLD;p.IMPACT=IMPACT
review.CASES=CASES;review.API=API

def evaluate(suite,case):
 correct=p.run_suite(suite,case); killed=[]
 if correct['pass']:
  for f in CASES[case]['faults']:
   if not p.run_suite(suite,case,fault=f)['pass']:killed.append(f)
 old={t['id']:t for t in OLD};actual={t['id']:t for t in suite};changed={k for k,v in old.items() if actual.get(k)!=v}
 return dict(correct_pass=correct['pass'],passed_tests=correct['passed'],total_tests=len(suite),killed=killed,new_faults_killed=len(set(killed)&{'rule_omitted','rule_incorrect'}),new_faults_total=2,regression_faults_killed=int('regression_detail_body' in killed),regression_faults_total=1,changed_existing=len(changed),unnecessary_changes=len(changed-IMPACT[case]),added_tests=len(set(actual)-set(old)),failures=correct['failures'])
review.evaluate=evaluate

def smoke():
 rows=[]
 for case in CASES:
  old=p.run_suite(OLD,case,new=False); stale=p.run_suite(OLD,case); e=evaluate(reference(case),case)
  assert old['pass'] and e['correct_pass'] and len(e['killed'])==3,(case,old,e)
  assert {x['id'] for x in stale['failures']}==IMPACT[case],(case,stale)
  rows.append(dict(case=case,old_tests=old['passed'],stale_passed=stale['passed'],reference_tests=e['total_tests'],reference_passed=e['passed_tests'],faults_killed=len(e['killed'])))
 return rows

METHODS=('A','B','C','D','C2','D2')
MODEL='gemini-3.5-flash-lite'

def base_run(case,method,ask,max_repairs=2):
 public={k:v for k,v in CASES[case].items() if k!='faults'}
 common=API+'\nRequirements: '+json.dumps(public)+'\nReturn JSON {"tests": [...]} using DSL. Preserve stable IDs for existing tests; use new IDs for additions. Do not weaken checks.'
 selected=None;suite=None
 if method=='D':
  try:
   plan=ask(common+'\nExisting tests: '+json.dumps(OLD)+'\nFirst return JSON {"update_ids":[...],"reasons":{...}} identifying existing tests that need edits, not merely related tests. New tests can be added later.')
   ids=plan['update_ids'];assert isinstance(ids,list) and all(isinstance(x,str) for x in ids),'invalid impact IDs'
   selected=set(ids);assert selected<={t['id'] for t in OLD},'unknown IDs'
  except (ValueError,KeyError,TypeError,AssertionError,AttributeError):return None,None
  allowed=[t for t in OLD if t['id'] in selected];locked=[t for t in OLD if t['id'] not in selected]
  base=common+'\nEditable tests: '+json.dumps(allowed)+'\nProtected tests (read-only): '+json.dumps(locked)+'\nReturn only updated editable tests (all selected IDs) and any added tests, never protected IDs.'
  def merge(data):
   changed=data['tests'];assert isinstance(changed,list)
   if changed:p.validate(changed)
   ids={t['id'] for t in changed};assert selected<=ids and not ids.intersection(t['id'] for t in locked),'protected or missing IDs'
   return p.validate(locked+changed)
 else:
  base=common+ ('\nRegenerate entire suite from requirements, without old tests.' if method=='A' else '\nExisting tests: '+json.dumps(OLD)+'\nUpdate complete suite directly; preserve unaffected behavior.')
  def merge(data):return p.validate(data['tests'])
 for _ in range(1+(max_repairs if method in ('C','D') else 0)):
  try:
   data=ask(base);suite=merge(data)
   if method not in ('C','D'):break
   result=p.run_suite(suite,case)
   if result['pass']:break
   base+='\nPrevious output: '+json.dumps(data)+'\nCorrect-version failures: '+json.dumps(result['failures'])+'\nRepair only legitimate issues; follow original constraints.'
  except (ValueError,KeyError,TypeError,AssertionError,AttributeError) as e:
   suite=None;base+='\nInvalid previous output: '+str(e)+'\nReturn valid JSON in required schema.'
 return suite,selected

def atomic(path,data):
 tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(data,indent=2));tmp.replace(path)
def collect(out):
 return [json.loads(f.read_text())['metrics'] for f in sorted(Path(out).glob('F*_*.json'))]
def export(out):
 out=Path(out);rows=collect(out)
 if rows:
  with (out/'results.csv').open('w',newline='') as f:
   w=csv.DictWriter(f,fieldnames=sorted(set().union(*(r.keys() for r in rows))));w.writeheader();w.writerows(rows)
  atomic(out/'results.json',rows)
 return rows

def run_batch(client,out,case_ids,repeats=(1,),model=MODEL,max_repairs=2):
 """Checkpoint each completed trial; abort on provider error; rerun resumes missing trials.
 C2/D2 share the exact base suites; invalid base earns zero, not a fresh retry.
 """
 assert model==MODEL,'fixed model required';out=Path(out);out.mkdir(parents=True,exist_ok=True)
 manifest=dict(model=model,max_repairs=max_repairs,methods=list(METHODS),benchmark='XP-final-synthetic-v1')
 mf=out/'manifest.json'
 if mf.exists():assert json.loads(mf.read_text())==manifest,'different configuration'
 else:atomic(mf,manifest)
 for rep in repeats:
  for case in case_ids:
   assert case in CASES
   order=list(('A','B','C','D'));random.Random(f'XP:{case}:{rep}').shuffle(order)
   for method in order+['C2','D2']:
    dest=out/f'{case}_{method}_{rep}.json'
    if dest.exists():continue
    logs=[];start=time.perf_counter();selected=None;obligations=[];attempts=[];status='evaluated';parent=None
    def ask(prompt):return p.model_call(client,model,prompt,logs)
    try:
     if method in ('C2','D2'):
      parent=json.loads((out/f'{case}_{method[0]}_{rep}.json').read_text());suite=parent['suite'];selected=parent['selected']
      if not parent['metrics']['correct_pass']:status='invalid_base'
      else:
       suite,obligations,status,attempts=review.review_case(case,suite,ask,max_repairs)
       if status=='provider_or_runner_error':raise RuntimeError('Coverage review interrupted by provider or runner; resume this trial.')
     else:suite,selected=base_run(case,method,ask,max_repairs)
    except Exception as e:
     # Keep provider failures out of scientific scores. Preserve partial calls for diagnosis.
     atomic(out/'interrupted.json',dict(case=case,method=method,repeat=rep,error_type=type(e).__name__,calls=logs));export(out);raise
    elapsed=time.perf_counter()-start;tokens=sum(x['tokens'] or 0 for x in logs);calls=len(logs)
    metrics=evaluate(suite,case) if suite is not None else dict(correct_pass=False,total_tests=0,new_faults_killed=0,new_faults_total=2,regression_faults_killed=0,regression_faults_total=1,changed_existing=None,unnecessary_changes=None,added_tests=None)
    if method=='A':
     for k in ('changed_existing','unnecessary_changes','added_tests'):metrics[k]=None
    if selected is not None:
     sel=set(selected);tp=len(sel&IMPACT[case]);metrics.update(impact_precision=tp/len(sel) if sel else (1.0 if not IMPACT[case] else 0.0),impact_recall=tp/len(IMPACT[case]) if IMPACT[case] else 1.0)
    if suite is None:status='invalid_output'
    metrics.update(case=case,repeat=rep,method=method,model=model,status=status,seconds=elapsed+(parent['metrics']['seconds'] if parent else 0),tokens=tokens+(parent['metrics']['tokens'] if parent else 0),calls=calls+(parent['metrics']['calls'] if parent else 0),review_seconds=elapsed if parent else 0,review_tokens=tokens if parent else 0,review_calls=calls if parent else 0)
    atomic(dest,dict(metrics=metrics,suite=suite,selected=sorted(selected) if selected is not None else None,obligations=obligations,attempts=attempts,calls=logs));export(out);print(case,rep,method,status,flush=True)
 return export(out)


## 1. Validate the final benchmark (no API calls)

In [ ]:
import pandas as pd
import final_study as study
smoke_rows = study.smoke()
display(pd.DataFrame(smoke_rows))
print("Final benchmark validated: 12 cases, 36 seeded fault variants.")

## 2. Save or restore local checkpoints
Use a dedicated local Colab folder. Completed trials persist within the current runtime only. Export a ZIP after every batch. To resume in a new runtime, upload a previous checkpoint ZIP when prompted (set RESTORE_CHECKPOINT = True before running this cell). Do not restore into an existing nonempty checkpoint folder. No API key is stored.


In [ ]:
from pathlib import Path
import zipfile, re
OUT = Path('/content/XP_AcceptanceAI_Final_Study')
OUT.mkdir(parents=True, exist_ok=True)
RESTORE_CHECKPOINT = False
if RESTORE_CHECKPOINT:
    from google.colab import files
    assert not any(OUT.iterdir()), "Restore into an empty folder only."
    uploaded = files.upload()
    assert len(uploaded) == 1, "Select one checkpoint ZIP."
    import io
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
        assert z.testzip() is None, "Corrupt ZIP."
        assert 'manifest.json' in z.namelist(), "Missing manifest."
        for name in z.namelist():
            if name in ('manifest.json', 'results.csv', 'results.json', 'interrupted.json') or re.fullmatch(r'F(0[1-9]|1[0-2])_(A|B|C|C2|D|D2)_[123]\.json', name):
                (OUT / name).write_bytes(z.read(name))
print("Results folder:", OUT)
print("Completed trials:", len(study.collect(OUT)))


## 3. Connect to Gemini (same API key)
Free-tier access depends on your account quota. Do not paste keys into source code. No extra connectivity test request is needed.

In [ ]:
import sys, subprocess, getpass
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
from openai import OpenAI
client = OpenAI(api_key=getpass.getpass('Gemini API key: '),
                base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
                max_retries=0, timeout=120)
print("Client ready; no API request made yet.")

## 4. Run a batch
Start with F01, repeat 1. Change only CASE_IDS and REPEAT_IDS after inspecting the result. If the provider quota stops the run, completed JSON records remain in the local checkpoint folder and a rerun skips them. An interrupted trial is not counted as a model failure; its partial calls are logged separately. Invalid generated suites receive zero fault-detection credit. Keep the same model and repair budget. Export after each batch. Provider 429/503 errors are retried at most twice; other errors are raised.

In [ ]:
import time
CASE_IDS = ["F01"]
REPEAT_IDS = [1]
# Preserve the original scientific protocol; retry only interrupted provider calls.
original_review = study.review.review_case
def review_with_status(case, suite, ask, max_repairs=2):
    errors = []
    def tracked_ask(prompt):
        try:
            return ask(prompt)
        except Exception as exc:
            errors.append(exc)
            raise
    result = original_review(case, suite, tracked_ask, max_repairs)
    if result[2] == "provider_or_runner_error" and errors:
        raise errors[-1]
    return result
study.review.review_case = review_with_status
try:
    for attempt in range(3):
        try:
            rows = study.run_batch(client, OUT, CASE_IDS, REPEAT_IDS)
            display(pd.DataFrame(rows).drop(columns=['failures','killed'], errors='ignore'))
            print("Completed trials:", len(rows), "/ 216 target")
            break
        except Exception as exc:
            if getattr(exc, "status_code", None) not in (429, 503):
                raise
            if attempt == 2:
                print("Gemini still unavailable. Export completed checkpoints before resuming.")
                break
            delay = 120 * (2 ** attempt)
            print(f"Gemini busy. Retrying in {delay} seconds...")
            time.sleep(delay)
finally:
    study.review.review_case = original_review
    print("Saved trials:", len(study.collect(OUT)))


## 5. Inspect and export completed results
Partial summaries are not final comparisons. Compare identical case/repeat pairs only; do not compare incomplete groups as if balanced.

In [ ]:
import shutil
from google.colab import files
rows = study.export(OUT)
if rows:
    df = pd.DataFrame(rows)
    display(df.groupby('method').agg(trials=('case','count'), correct_rate=('correct_pass','mean'), new_killed=('new_faults_killed','sum'), new_total=('new_faults_total','sum'), regression_killed=('regression_faults_killed','sum'), regression_total=('regression_faults_total','sum'), mean_seconds=('seconds','mean'), mean_tokens=('tokens','mean')))
    archive = shutil.make_archive('/content/xp_final_results', 'zip', str(OUT))
    files.download(archive)
else:
    print('No completed live trials yet.')